In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog", "catalog_project")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("storage-account", "adlssmartdata1921")

In [0]:
container = dbutils.widgets.get("container")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storage_account = dbutils.widgets.get("storage-account")

file_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/ppe_000_062026_1.csv"

In [0]:
fileppe = StructType(fields=[StructField("idTransaction", StringType(), False),
                                  StructField("cardNumber", StringType(), False),
                                  StructField("account", StringType(), False),
                                  StructField("dateppe", StringType(), False),
                                  StructField("amount", StringType(), False)
                                  ])

In [0]:

filepperead = spark.read.option('header', True)\
                           .schema(fileppe)\
                           .csv(file_path)

In [0]:
fileppereaddt = filepperead.withColumn("dateppe",to_date(col("dateppe"), "dd/MM/yyyy"))
fileppeingestiondate = fileppereaddt.withColumn("ingestion_date", current_timestamp())


In [0]:
df_fileppe = fileppeingestiondate.select("idTransaction",
                                                 "cardNumber",
                                                 "account",
                                                 "dateppe",
                                                 "amount",
                                                 "ingestion_date"
                                                 )

In [0]:

df_fileppe.write.mode("overwrite").insertInto(f"{catalog}.{schema}.fileppe")